# Version B EDA: Collaborative Filtering Data Analysis

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

In [ ]:
DATA_DIR = "../../Data/Pure_Data"
FIGURES_DIR = "Figures"
SUMMARY_PATH = "../Results/version_b_eda_summary.csv"

required_files = [
    "interactions_filtered.csv",
    "interactions_train_filtered.csv",
    "interactions_test_filtered.csv",
]

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

for f in required_files:
    p = os.path.join(DATA_DIR, f)
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing required file: {p}")

os.makedirs(FIGURES_DIR, exist_ok=True)

interactions_filtered = pd.read_csv(os.path.join(DATA_DIR, "interactions_filtered.csv"))
train = pd.read_csv(os.path.join(DATA_DIR, "interactions_train_filtered.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "interactions_test_filtered.csv"))

for name, df in [("interactions_filtered", interactions_filtered), ("train", train), ("test", test)]:
    for col in ["user_id", "recipe_id", "rating"]:
        if col not in df.columns:
            raise ValueError(f"{name} missing required column: {col}")

print("interactions_filtered:", interactions_filtered.shape)
print("train:", train.shape)
print("test:", test.shape)
display(interactions_filtered.head())

In [ ]:
summary = {}
summary["filtered_interactions"] = int(len(interactions_filtered))
summary["train_interactions"] = int(len(train))
summary["test_interactions"] = int(len(test))
summary["filtered_users"] = int(interactions_filtered["user_id"].nunique())
summary["filtered_recipes"] = int(interactions_filtered["recipe_id"].nunique())

summary_df = pd.DataFrame([{"metric":k, "value":v} for k,v in summary.items()])
display(summary_df)

In [ ]:
# 1) version_b_rating_distribution.png
cnt = interactions_filtered["rating"].value_counts().sort_index()
plt.figure(figsize=(7,4))
plt.bar(cnt.index.astype(str), cnt.values)
plt.title("Version B Rating Distribution")
plt.xlabel("rating")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_rating_distribution.png"), dpi=150)
plt.close()

# keep legacy name for compatibility
plt.figure(figsize=(7,4))
plt.bar(cnt.index.astype(str), cnt.values)
plt.title("Rating Distribution")
plt.xlabel("rating")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rating_distribution.png"), dpi=150)
plt.close()

In [ ]:
# 2) version_b_user_activity_distribution.png
user_cnt = interactions_filtered.groupby("user_id").size()
u_q99 = float(user_cnt.quantile(0.99))
plt.figure(figsize=(8,4.5))
plt.hist(user_cnt.values, bins=50)
plt.xlim(0, u_q99)
plt.title("Version B User Activity Distribution (clipped at 99th percentile)")
plt.xlabel("interactions per user")
plt.ylabel("frequency")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_user_activity_distribution.png"), dpi=150)
plt.close()

# keep legacy name for compatibility
plt.figure(figsize=(8,4.5))
plt.hist(user_cnt.values, bins=50)
plt.xlim(0, u_q99)
plt.title("User Interaction Distribution")
plt.xlabel("interactions per user")
plt.ylabel("frequency")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "user_interaction_distribution.png"), dpi=150)
plt.close()

# 3) version_b_recipe_popularity_distribution.png
recipe_cnt = interactions_filtered.groupby("recipe_id").size()
r_q99 = float(recipe_cnt.quantile(0.99))
plt.figure(figsize=(8,4.5))
plt.hist(recipe_cnt.values, bins=50)
plt.xlim(0, r_q99)
plt.title("Version B Recipe Popularity Distribution (clipped at 99th percentile)")
plt.xlabel("interactions per recipe")
plt.ylabel("frequency")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_recipe_popularity_distribution.png"), dpi=150)
plt.close()

# keep legacy name for compatibility
plt.figure(figsize=(8,4.5))
plt.hist(recipe_cnt.values, bins=50)
plt.xlim(0, r_q99)
plt.title("Recipe Interaction Distribution")
plt.xlabel("interactions per recipe")
plt.ylabel("frequency")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "recipe_interaction_distribution.png"), dpi=150)
plt.close()

In [ ]:
# 4) version_b_matrix_sparsity_summary.png + summary stats
num_users = interactions_filtered["user_id"].nunique()
num_recipes = interactions_filtered["recipe_id"].nunique()
num_interactions = len(interactions_filtered)
possible_cells = num_users * num_recipes
density = num_interactions / possible_cells if possible_cells else np.nan
sparsity = 1 - density if possible_cells else np.nan

summary["matrix_possible_cells"] = int(possible_cells)
summary["matrix_density"] = float(density)
summary["matrix_sparsity"] = float(sparsity)

plt.figure(figsize=(8,4.5))
labels = ["users", "recipes", "interactions", "density", "sparsity"]
vals = [num_users, num_recipes, num_interactions, density, sparsity]
plt.bar(labels, vals)
plt.title("Version B Matrix Sparsity Summary")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_matrix_sparsity_summary.png"), dpi=150)
plt.close()

# 5) history buckets
user_bucket = pd.cut(user_cnt, bins=[0,1,4,9,19,user_cnt.max()], labels=["1","2-4","5-9","10-19","20+"])
bucket_counts = user_bucket.value_counts().sort_index()
plt.figure(figsize=(8,4.5))
plt.bar(bucket_counts.index.astype(str), bucket_counts.values)
plt.title("Version B History Bucket Overview (Users)")
plt.xlabel("train interaction bucket")
plt.ylabel("user count")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_history_bucket_overview.png"), dpi=150)
plt.close()

In [ ]:
# 6) train/test overlap summary
train_users = set(train["user_id"].unique())
test_users = set(test["user_id"].unique())
train_recipes = set(train["recipe_id"].unique())
test_recipes = set(test["recipe_id"].unique())

cold_users = test_users - train_users
cold_recipes = test_recipes - train_recipes

summary["test_users"] = int(len(test_users))
summary["test_users_in_train"] = int(len(test_users & train_users))
summary["cold_start_test_users"] = int(len(cold_users))
summary["test_recipes"] = int(len(test_recipes))
summary["test_recipes_in_train"] = int(len(test_recipes & train_recipes))
summary["cold_start_test_recipes"] = int(len(cold_recipes))

plt.figure(figsize=(8,4.5))
labels = ["test_users", "users_in_train", "cold_users", "test_recipes", "recipes_in_train", "cold_recipes"]
vals = [len(test_users), len(test_users & train_users), len(cold_users), len(test_recipes), len(test_recipes & train_recipes), len(cold_recipes)]
plt.bar(labels, vals)
plt.xticks(rotation=20, ha="right")
plt.title("Version B Train/Test Overlap Summary")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_train_test_overlap_summary.png"), dpi=150)
plt.close()

In [ ]:
# 7) positive feedback ratio
thr = 4
ratios = {
    "filtered_positive_ratio": float((interactions_filtered["rating"] >= thr).mean()),
    "train_positive_ratio": float((train["rating"] >= thr).mean()),
    "test_positive_ratio": float((test["rating"] >= thr).mean()),
}
summary.update(ratios)

plt.figure(figsize=(7,4.5))
plt.bar(list(ratios.keys()), list(ratios.values()))
plt.ylim(0,1)
plt.title("Version B Positive Feedback Ratio (rating >= 4)")
plt.ylabel("ratio")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_positive_feedback_ratio.png"), dpi=150)
plt.close()

In [ ]:
# 8) long-tail recipe distribution
rc = recipe_cnt.sort_values(ascending=False).reset_index(drop=True)
cum = rc.cumsum() / rc.sum()
x = np.arange(1, len(rc)+1)

plt.figure(figsize=(8,4.5))
plt.plot(x, cum)
plt.title("Version B Long-tail Recipe Distribution")
plt.xlabel("recipes sorted by interaction count")
plt.ylabel("cumulative interaction share")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "version_b_long_tail_recipe_distribution.png"), dpi=150)
plt.close()

summary_df = pd.DataFrame([{"metric":k, "value":v} for k,v in summary.items()])
summary_df.to_csv(SUMMARY_PATH, index=False)
display(summary_df)

## EDA Takeaways

- The filtered subset supports collaborative-filtering experiments on a non-trivial user-item graph.
- The user-item matrix is still highly sparse.
- User activity and recipe popularity are strongly long-tailed.
- Positive feedback (`rating >= 4`) is very common in this filtered split.
- Cold-start users/recipes in filtered temporal test are expected to be near zero when overlap checks pass.
- These observations motivate evaluating kNN and matrix factorization in Version B, and later hybrid integration.